# Shape Data Experiments

This notebook explores how random projections with ReLU activations affect 2D geometric shapes.

**Experiments:**
1. Compare PCA vs Random Projections on shapes
2. Multi-layer RP + ReLU transformations
3. Effect of shape position (negative vs positive coordinates)

In [ ]:
# Setup - add src to path if running from notebooks folder
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

from rp_study.data.shapes import (
    generate_circle, generate_ellipse, generate_square, generate_rectangle,
    rotate_shape, generate_standard_shapes
)
from rp_study.projections import multi_layer_projection, apply_random_projection
from rp_study.visualization.projection_plots import plot_shape_transformations

# Set random seed for reproducibility
np.random.seed(42)

## Configuration

Modify these parameters to customize the experiments:

In [ ]:
# Shape generation parameters
N_POINTS = 200  # Number of points per shape

# Multi-layer experiment parameters
LAYER_COUNTS = [1, 2, 3, 5, 10, 20]  # Number of RP+ReLU layers to test

## 1. Generate Shapes

In [ ]:
# Generate shapes with negative coordinates (original centers)
shapes_neg = generate_standard_shapes(n_points=N_POINTS, with_negative_coords=True)

# Generate shapes with positive coordinates (shifted)
shapes_pos = generate_standard_shapes(n_points=N_POINTS, with_negative_coords=False)

# Visualize original shapes
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, (name, X) in enumerate(shapes_neg.items()):
    axes[0, i].scatter(X[:, 0], X[:, 1], s=15)
    axes[0, i].set_title(f"{name} (negative coords)")
    axes[0, i].axis('equal')

for i, (name, X) in enumerate(shapes_pos.items()):
    axes[1, i].scatter(X[:, 0], X[:, 1], s=15)
    axes[1, i].set_title(f"{name} (positive coords)")
    axes[1, i].axis('equal')

plt.suptitle("Original Shapes", fontsize=14)
plt.tight_layout()
plt.show()

## 2. PCA vs Random Projection

Compare how PCA and Random Projection transform the shapes, including the effect of ReLU activation.

In [ ]:
def relu(data):
    """Apply ReLU activation."""
    return np.maximum(0, data)

def plot_pca_rp_comparison(shapes, title_prefix):
    """Plot PCA vs RP comparison for a set of shapes."""
    fig, axes = plt.subplots(4, 4, figsize=(16, 16))
    
    for i, (name, X) in enumerate(shapes.items()):
        # PCA
        pca = PCA(n_components=2)
        X_pca = pca.fit_transform(X)
        
        # Random Projection
        d = X.shape[1]
        a = np.sqrt(3 * (2/d))
        R = np.random.uniform(-a, a, size=(d, 2))
        X_rp = X @ R
        
        # Apply ReLU
        X_pca_relu = relu(X_pca)
        X_rp_relu = relu(X_rp)
        
        # Plot
        axes[i, 0].scatter(X_pca[:, 0], X_pca[:, 1], s=15)
        axes[i, 0].set_title(f"{name} - PCA")
        axes[i, 0].axis('equal')
        
        axes[i, 1].scatter(X_rp[:, 0], X_rp[:, 1], s=15)
        axes[i, 1].set_title(f"{name} - RP")
        axes[i, 1].axis('equal')
        
        axes[i, 2].scatter(X_pca_relu[:, 0], X_pca_relu[:, 1], s=15)
        axes[i, 2].set_title(f"{name} - PCA+ReLU")
        axes[i, 2].axis('equal')
        
        axes[i, 3].scatter(X_rp_relu[:, 0], X_rp_relu[:, 1], s=15)
        axes[i, 3].set_title(f"{name} - RP+ReLU")
        axes[i, 3].axis('equal')
    
    plt.suptitle(f"{title_prefix}: PCA vs Random Projection", fontsize=14)
    plt.tight_layout()
    plt.show()

# Compare with negative coordinates
plot_pca_rp_comparison(shapes_neg, "Negative Coords")

In [ ]:
# Compare with positive coordinates
plot_pca_rp_comparison(shapes_pos, "Positive Coords")

## 3. Multi-Layer RP + ReLU Transformations

Apply multiple layers of random projections with ReLU and observe the transformation.

In [ ]:
def apply_layers(X, num_layers):
    """Apply multiple layers of RP + ReLU."""
    X_layer = X.copy()
    for _ in range(num_layers):
        d = X_layer.shape[1]
        a = np.sqrt(3 * (2/d))
        R = np.random.uniform(-a, a, size=(d, d))
        X_layer = relu(X_layer @ R)
    return X_layer

# Apply multi-layer transformations
for num_layers in LAYER_COUNTS:
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle(f"Data after {num_layers} RP-ReLU Layers", fontsize=16)
    
    for i, (name, X) in enumerate(shapes_neg.items()):
        X_out = apply_layers(X, num_layers)
        axes[0, i].scatter(X_out[:, 0], X_out[:, 1], s=15)
        axes[0, i].set_title(f"{name} (neg)")
        axes[0, i].axis('equal')
    
    for i, (name, X) in enumerate(shapes_pos.items()):
        X_out = apply_layers(X, num_layers)
        axes[1, i].scatter(X_out[:, 0], X_out[:, 1], s=15)
        axes[1, i].set_title(f"{name} (pos)")
        axes[1, i].axis('equal')
    
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

## 4. Tilted Shapes Experiment

Apply rotation to shapes before transformation.

In [ ]:
# Create tilted versions of shapes
TILT_ANGLE = 45  # degrees

shapes_tilted = {}
for name, X in shapes_neg.items():
    shapes_tilted[name] = rotate_shape(X, TILT_ANGLE)

# Visualize tilted shapes
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, (name, X) in enumerate(shapes_neg.items()):
    axes[0, i].scatter(X[:, 0], X[:, 1], s=15)
    axes[0, i].set_title(f"{name} (original)")
    axes[0, i].axis('equal')

for i, (name, X) in enumerate(shapes_tilted.items()):
    axes[1, i].scatter(X[:, 0], X[:, 1], s=15)
    axes[1, i].set_title(f"{name} (tilted {TILT_ANGLE}°)")
    axes[1, i].axis('equal')

plt.suptitle("Original vs Tilted Shapes", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Apply RP+ReLU to tilted shapes
for num_layers in [1, 5, 10]:
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    fig.suptitle(f"Tilted Shapes after {num_layers} RP-ReLU Layers", fontsize=14)
    
    for i, (name, X) in enumerate(shapes_tilted.items()):
        X_out = apply_layers(X, num_layers)
        axes[i].scatter(X_out[:, 0], X_out[:, 1], s=15)
        axes[i].set_title(name)
        axes[i].axis('equal')
    
    plt.tight_layout()
    plt.show()

## Summary

Key observations:
1. Random projections preserve geometric structure differently than PCA
2. ReLU activation clips negative values, dramatically changing shapes with negative coordinates
3. Multiple layers of RP+ReLU progressively transform the data
4. Initial position (negative vs positive coordinates) affects the outcome due to ReLU